In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from tqdm import tqdm
from scipy.stats import pearsonr
from collections import defaultdict
import gseapy as gp
import anndata
import pandas as pd
from scipy import stats
from evaluation.eval_utils import *
# from plot_utils import *
from adjustText import adjust_text
from matplotlib.colors import LinearSegmentedColormap
#from plottable import ColumnDefinition, Table
#from plottable.cmap import normed_cmap
#from plottable.formatters import decimal_to_percent
#from plottable.plots import bar, percentile_bars, percentile_stars, progress_donut
import matplotlib
import os

sns.set_style('whitegrid')

In [2]:
outdir = '../results'
figdir = '../fig'


datasets = [
            'TianCRISPRa2021', 
            'TianCRISPRi2021',
            'ReplogleRPE1_v2',
            'ReplogleK562_gwps',
            'XuKinetics2024',
            'FrangiehControlSingle2021', 'FrangiehCocultureSingle2021', 'FrangiehIfnSingle2021',
            'Norman2019',
            'Adamson2016'
           ]
dataset_names = [
                 'tian_CRISPRa_2021_scperturb',
                  'tian_CRISPRi_2021_scperturb',
                  'replogle_rpe1_v2_2022',
                  'replogle_k562_gwps_2022',
                 'xu_kinetics_2024',
                  'frangieh_control_single_2021',
                 'frangieh_coculture_single_2021',
                 'frangieh_ifn_single_2021',
                 'norman2019',
                 'adamson2016'
                ]
dataset_labels = [
                  'Tian\n(CRISPRa)', 
                  'Tian\n(CRISPRi)',
                   'Replogle\n(RPE1)',
                   'Replogle\n(K562)',
                   'Xu\n(Kinetics)',
                   'Frangieh\n(Control)', 'Frangieh\n(Co-culture)', 'Frangieh\n(IFN)',
                  'Norman',
                  'Adamson'
                 ]


dataset_dict = {k: v for k, v in zip(dataset_names, datasets)}
dataset_label_dict = {k: v for k, v in zip(dataset_names, dataset_labels)}
dataset_map = {k: v for k, v in zip(datasets, dataset_labels)}

seeds = [1, 2, 3]
n_control_cells = None # Set to an integer to use that many, or None for all
control_split = False
methods = ['nonctl-mean']

In [3]:
results_df = pd.DataFrame(
    columns=["dataset", "method", "pert", "seed", "corr_all", "corr_20de",
             "corr_all_allpert", "corr_20de_allpert",
             "mse_all", "mse_20de", "mse_all_allpert", "mse_20de_allpert",
             "jaccard", "jaccard_allpert", "one gene", "train"]
)
avg_pert_centroids = 'centroids'

np.random.seed(42)

for dataset, dataset_name in zip(datasets, dataset_names):
    for seed in seeds:
        file = f'../data/{dataset_name}/{dataset_name}_{seed}.h5ad'
        adata = anndata.read_h5ad(file)
        print(dataset, len(adata.obs['condition'].unique()))
        
        # Get control mean, non control mean (pert_mean), and non control mean differential
        train_adata = adata[adata.obs['split'] == 'train']
        control_adata = train_adata[train_adata.obs['control'] == 1]
        pert_adata = train_adata[train_adata.obs['control'] == 0]

        if 'Frangieh' in dataset:
            control_cells = adata[adata.obs['sgRNAs'].str.contains('_SITE_')]
        else:
            control_cells = control_adata

        control_cells_original = control_cells

        
        #Print length of control cell 
        print("Number of control cells: ", len(control_cells))
        print("Dimension of train adata ", train_adata.shape)

        control_mean = np.array(control_cells.X.mean(axis=0))[0]
        if avg_pert_centroids == 'centroids':
            pert_mean = average_of_perturbation_centroids(pert_adata)
        else:
            pert_mean = np.array(pert_adata.X.mean(axis=0))[0]
        
        for method in tqdm(methods):
            p = f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv'
            if not os.path.exists(p):
                print('Predictions not ready for method: ', method, 'dataset: ', dataset, 'seed: ', seed)
                continue
            post_gt_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-gt.csv', index_col=[0, 1])
            post_pred_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv', index_col=[0, 1])
            conditions = post_gt_df.index.get_level_values('condition').unique()
            for condition in conditions:
                gene_list = condition.split("+")
                one_gene = False
                if "ctrl" in gene_list:
                    gene_list.remove("ctrl")
                    one_gene = True
                one_gene_str = "1-gene" if one_gene else "2-gene"
    
                # Get data
                X_true = post_gt_df.loc[condition].values[0]
                X_pred = post_pred_df.loc[condition].values[0]

                n_control = len(control_cells)
                # Split control cells into two groups
                n_group1 = n_control // 2
                n_group2 = n_control - n_group1

                # Randomly shuffle indices and split
                control_indices = np.arange(n_control)
                np.random.shuffle(control_indices)
                group1_indices = control_indices[:n_group1]
                group2_indices = control_indices[n_group1:]

                # Create two groups
                group1_cells = control_cells[group1_indices]
                group2_cells = control_cells[group2_indices]

                # Compute means for each group
                control_mean1 = np.array(group1_cells.X.mean(axis=0))[0]
                control_mean2 = np.array(group2_cells.X.mean(axis=0))[0]

                #Now control_mean1 is the reference
                #control_mean2 is the prediction of perturbed 
                delta_true = X_true - control_mean2
                delta_pred = control_mean1 - control_mean2

                delta_true_allpert = X_true - pert_mean
                delta_pred_allpert = X_pred - pert_mean
                n_train = post_gt_df.loc[condition].index.get_level_values('n_train').values[0]

                # Get top 20 DE genes
                adata_condition = adata[adata.obs["condition"] == condition]

                # Select top 20 DE genes
                top20_de_genes = adata.uns["top_non_dropout_de_20"][
                    adata_condition.obs["condition_name"].values[0]
                ]
                top20_de_idxs = np.argwhere(
                    np.isin(adata.var.index, top20_de_genes)
                ).ravel()

                top20_de_idxs_pred = get_topk_de_gene_ids(control_mean, X_pred, k=20)

                # Store results
                results_df.loc[len(results_df)] = [
                    dataset,
                    method,
                    condition,
                    seed,
                    pearsonr(delta_true, delta_pred)[0],
                    pearsonr(delta_true[top20_de_idxs], delta_pred[top20_de_idxs])[0],
                    pearsonr(delta_true_allpert, delta_pred_allpert)[0],
                    pearsonr(delta_true_allpert[top20_de_idxs], delta_pred_allpert[top20_de_idxs])[0],
                    np.mean((delta_true - delta_pred)**2),
                    np.mean((delta_true[top20_de_idxs] - delta_pred[top20_de_idxs])**2),
                    np.mean((delta_true_allpert - delta_pred_allpert)**2),  # NOTE: This metric is independent of the reference
                    np.mean((delta_true_allpert[top20_de_idxs] - delta_pred_allpert[top20_de_idxs])**2),  # NOTE: This metric is independent of the reference
                    jaccard_similarity(top20_de_idxs, top20_de_idxs_pred),
                    np.nan, # jaccard_similarity(top20_de_idxs_allpert, top20_de_idxs_pred_allpert),
                    one_gene_str,
                    n_train,
                ]

TianCRISPRa2021 98
Number of control cells:  434
Dimension of train adata  (13567, 5052)


100%|██████████| 1/1 [00:00<00:00,  1.10it/s]


TianCRISPRa2021 98
Number of control cells:  434
Dimension of train adata  (13189, 5052)


100%|██████████| 1/1 [00:00<00:00,  1.16it/s]


TianCRISPRa2021 98
Number of control cells:  434
Dimension of train adata  (14195, 5052)


...

100%|██████████| 1/1 [00:02<00:00,  2.12s/it]


Adamson2016 82
Number of control cells:  24263
Dimension of train adata  (52593, 5060)


100%|██████████| 1/1 [00:02<00:00,  2.21s/it]


Adamson2016 82
Number of control cells:  24263
Dimension of train adata  (52173, 5060)


100%|██████████| 1/1 [00:02<00:00,  2.15s/it]


In [ ]:
#Save results_df to ../results
results_df.to_csv(f'{outdir}/ctl-mean-results.csv', index=False)


#Compute average correlation
avg_corrs = results_df.groupby(['dataset', 'method'])['corr_all'].mean().reset_index()
avg_corrs 

In [6]:
results_df = pd.DataFrame(
    columns=["dataset", "method", "pert", "seed", "corr_all", "corr_20de",
             "corr_all_allpert", "corr_20de_allpert",
             "mse_all", "mse_20de", "mse_all_allpert", "mse_20de_allpert",
             "jaccard", "jaccard_allpert", "one gene", "train"]
)
avg_pert_centroids = 'centroids'
np.random.seed(42)
for dataset, dataset_name in zip(datasets, dataset_names):
    for seed in seeds:
        file = f'../data/{dataset_name}/{dataset_name}_{seed}.h5ad'
        adata = anndata.read_h5ad(file)
        print(dataset, len(adata.obs['condition'].unique()))
        
        # Get control mean, non control mean (pert_mean), and non control mean differential
        train_adata = adata[adata.obs['split'] == 'train']
        control_adata = train_adata[train_adata.obs['control'] == 1]
        pert_adata = train_adata[train_adata.obs['control'] == 0]

        if 'Frangieh' in dataset:
            control_cells = adata[adata.obs['sgRNAs'].str.contains('_SITE_')]
        else:
            control_cells = control_adata

        control_cells_original = control_cells

        
        #Print length of control cell 
        print("Number of control cells: ", len(control_cells))
        print("Dimension of train adata ", train_adata.shape)

        control_mean = np.array(control_cells.X.mean(axis=0))[0]
        if avg_pert_centroids == 'centroids':
            pert_mean = average_of_perturbation_centroids(pert_adata)
        else:
            pert_mean = np.array(pert_adata.X.mean(axis=0))[0]
        
        for method in tqdm(methods):
            p = f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv'
            if not os.path.exists(p):
                print('Predictions not ready for method: ', method, 'dataset: ', dataset, 'seed: ', seed)
                continue
            post_gt_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-gt.csv', index_col=[0, 1])
            post_pred_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv', index_col=[0, 1])
            conditions = post_gt_df.index.get_level_values('condition').unique()
            for condition in conditions:
                gene_list = condition.split("+")
                one_gene = False
                if "ctrl" in gene_list:
                    gene_list.remove("ctrl")
                    one_gene = True
                one_gene_str = "1-gene" if one_gene else "2-gene"
    
                # Get data
                X_true = post_gt_df.loc[condition].values[0]
                X_pred = post_pred_df.loc[condition].values[0]

                n_control = len(control_cells)
                # Split control cells into two groups
                n_group1 = n_control // 2
                n_group2 = n_control - n_group1

                # Randomly shuffle indices and split
                control_indices = np.arange(n_control)
                np.random.shuffle(control_indices)
                group1_indices = control_indices[:n_group1]
                group2_indices = control_indices[n_group1:]

                # Create two groups
                group1_cells = control_cells[group1_indices]
                group2_cells = control_cells[group2_indices]

                #Split group2 cells into group2_1 group2_2
                n_group21 = n_group2 // 2
                n_group22 = n_group2 - n_group21
                group21_indices = control_indices[n_group1:n_group1+n_group21]
                group22_indices = control_indices[n_group1+n_group21:]
                group21_cells = control_cells[group21_indices]
                group22_cells = control_cells[group22_indices]

                # Compute means for each group
                control_mean = np.array(control_cells.X.mean(axis=0))[0]
                control_mean1 = np.array(group1_cells.X.mean(axis=0))[0]
                control_mean21 = np.array(group21_cells.X.mean(axis=0))[0]
                control_mean22 = np.array(group22_cells.X.mean(axis=0))[0]

                #Now control_mean1 is the reference
                #control_mean2 is the prediction of perturbed 
                delta_true = X_true - control_mean21
                delta_pred = control_mean1 - control_mean22

                delta_true_allpert = X_true - pert_mean
                delta_pred_allpert = X_pred - pert_mean
                n_train = post_gt_df.loc[condition].index.get_level_values('n_train').values[0]

                # Get top 20 DE genes
                adata_condition = adata[adata.obs["condition"] == condition]

                # Select top 20 DE genes
                top20_de_genes = adata.uns["top_non_dropout_de_20"][
                    adata_condition.obs["condition_name"].values[0]
                ]
                top20_de_idxs = np.argwhere(
                    np.isin(adata.var.index, top20_de_genes)
                ).ravel()

                top20_de_idxs_pred = get_topk_de_gene_ids(control_mean, X_pred, k=20)

                # Store results
                results_df.loc[len(results_df)] = [
                    dataset,
                    method,
                    condition,
                    seed,
                    pearsonr(delta_true, delta_pred)[0],
                    pearsonr(delta_true[top20_de_idxs], delta_pred[top20_de_idxs])[0],
                    pearsonr(delta_true_allpert, delta_pred_allpert)[0],
                    pearsonr(delta_true_allpert[top20_de_idxs], delta_pred_allpert[top20_de_idxs])[0],
                    np.mean((delta_true - delta_pred)**2),
                    np.mean((delta_true[top20_de_idxs] - delta_pred[top20_de_idxs])**2),
                    np.mean((delta_true_allpert - delta_pred_allpert)**2),  # NOTE: This metric is independent of the reference
                    np.mean((delta_true_allpert[top20_de_idxs] - delta_pred_allpert[top20_de_idxs])**2),  # NOTE: This metric is independent of the reference
                    jaccard_similarity(top20_de_idxs, top20_de_idxs_pred),
                    np.nan, # jaccard_similarity(top20_de_idxs_allpert, top20_de_idxs_pred_allpert),
                    one_gene_str,
                    n_train,
                ]

TianCRISPRa2021 98
Number of control cells:  434
Dimension of train adata  (13567, 5052)


100%|██████████| 1/1 [00:01<00:00,  1.90s/it]


TianCRISPRa2021 98
Number of control cells:  434
Dimension of train adata  (13189, 5052)


100%|██████████| 1/1 [00:01<00:00,  1.34s/it]


TianCRISPRa2021 98
Number of control cells:  434
Dimension of train adata  (14195, 5052)


...

100%|██████████| 1/1 [00:04<00:00,  4.15s/it]


Adamson2016 82
Number of control cells:  24263
Dimension of train adata  (52593, 5060)


100%|██████████| 1/1 [00:04<00:00,  4.09s/it]


Adamson2016 82
Number of control cells:  24263
Dimension of train adata  (52173, 5060)


100%|██████████| 1/1 [00:04<00:00,  4.13s/it]


In [ ]:
#Save results_df to ../results
results_df.to_csv(f'{outdir}/ctl-mean-results_split.csv', index=False)

#Compute average correlation
avg_corrs = results_df.groupby(['dataset', 'method'])['corr_all'].mean().reset_index()
avg_corrs 